"""
 ## GEOAI PADDY vs NOT-PADDY CLASSIFICATION PIPELINE
 - Sensor     : PlanetScope (4-band: B, G, R, NIR — 3m resolution)
 - Season     : Rabi 2025-26 | District: Koraput | Block: Borigumma
 - Author     : Senior GeoAI / Remote Sensing Engineer
 - Strategy   : Single-image maximum feature extraction → ensemble ML

PIPELINE OVERVIEW
─────────────────
 1. Read PlanetScope GeoTIFF + UDM2-style nodata masking
 2. Compute 10 vegetation indices
 3. Extract GLCM texture features (6 metrics × multi-band)
 4. Compute local spatial statistics (mean, variance, std) — 3×3 & 5×5
 5. Edge / structure features (Sobel, Laplacian, morphological gradient)
 6. PCA components + band ratios + entropy filters
 7. Stack → flat feature matrix → align with training shapefile samples
 8. Train Random Forest + XGBoost with SMOTE balancing + Optuna HPO
 9. Probability threshold optimisation (F1-optimal)
10. Classify full raster → export classified GeoTIFF + probability raster
11. Feature importance plot + confusion matrix + accuracy report
"""

In [ ]:
pip install scikit-learn geopandas fiona rasterio joblib imbalanced-learn xgboost

In [1]:
# =====================================================================
# FIX PROJ + FULL GIS/ML PIPELINE
# =====================================================================

# ---------------------------------------------------------------------
# 1. FIX PROJ DATABASE ISSUE
# ---------------------------------------------------------------------

import os
import sys

# IMPORTANT:
# Set BEFORE importing geopandas / rasterio / pyproj

CONDA_ENV = os.path.dirname(sys.executable)

POSSIBLE_PROJ_PATHS = [
    os.path.join(CONDA_ENV, "Library", "share", "proj"),
    os.path.join(CONDA_ENV, "..", "Library", "share", "proj"),
    r"D:\miniconda3\envs\crop_ai_env\Library\share\proj",
]

proj_found = False

for p in POSSIBLE_PROJ_PATHS:
    p = os.path.abspath(p)

    if os.path.exists(os.path.join(p, "proj.db")):
        os.environ["PROJ_LIB"] = p
        os.environ["PROJ_DATA"] = p
        proj_found = True

        print(f"✅ PROJ DATABASE FOUND")
        print(f"PROJ PATH = {p}")
        break

if not proj_found:
    raise Exception("❌ proj.db NOT FOUND")

# ---------------------------------------------------------------------
# 2. IMPORTS
# ---------------------------------------------------------------------

import gc
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import geopandas as gpd
import rasterio
import joblib
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import xgboost as xgb

from pathlib import Path
from pyproj import CRS
from shapely.geometry import mapping

from rasterio.mask import mask as rio_mask
from rasterio.windows import Window
from rasterio.features import geometry_mask

from scipy.ndimage import (
    uniform_filter,
    sobel as sp_sobel,
    laplace as sp_laplace
)

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay
)

from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# ---------------------------------------------------------------------
# 3. TEST CRS
# ---------------------------------------------------------------------

print("\n🧪 Testing PROJ...")
print(CRS.from_epsg(4326))

print("✅ All imports successful")

# ---------------------------------------------------------------------
# 4. CONFIG
# ---------------------------------------------------------------------

CFG = {
    "raster_path": r"\\192.168.0.243\paddy_ai\Planet_Image_Rabi_2526\Koraput\Borigumma_Rabi_2526.tif",

    "shp_path": r"E:\Rabi_Paddy_2526\LOT1\Koraput\TS\Borigumma_TS_TS.shp",

    "class_field": "class",

    "out_dir": r"E:\Rabi_Paddy_2526\LOT1\Koraput\Output",

    "scale_factor": 10000.0,

    "nodata_val": 0,

    # MEMORY FIX
    "tile_size": 1024,

    "pad": 8,

    "win": 5,

    "n_rf": 150,

    "n_xgb": 150,

    "cv_folds": 5,

    "seed": 42,
}

OUT = Path(CFG["out_dir"])
OUT.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Output Folder:")
print(OUT)

# ---------------------------------------------------------------------
# 5. HELPERS
# ---------------------------------------------------------------------

def safe_div(a, b, fill=0.0):

    with np.errstate(invalid="ignore", divide="ignore"):

        return np.where(
            np.abs(b) > 1e-6,
            a / b,
            fill
        ).astype(np.float32)

# ---------------------------------------------------------------------
# 6. FEATURE ENGINEERING
# ---------------------------------------------------------------------

_FEAT_NAMES = None

def compute_features(chip4, win=5):

    global _FEAT_NAMES

    B = chip4[0]
    G = chip4[1]
    R = chip4[2]
    N = chip4[3]

    feats = []
    names = []

    def add(arr, nm):

        feats.append(
            np.nan_to_num(arr, nan=0.0).astype(np.float32)
        )

        names.append(nm)

    # -------------------------------------------------------------
    # RAW BANDS
    # -------------------------------------------------------------

    for arr, nm in zip(
        [B, G, R, N],
        ["Blue", "Green", "Red", "NIR"]
    ):
        add(arr, nm)

    # -------------------------------------------------------------
    # VEGETATION INDICES
    # -------------------------------------------------------------

    NDVI = safe_div(N - R, N + R)

    GNDVI = safe_div(N - G, N + G)

    SAVI = safe_div(
        1.5 * (N - R),
        N + R + 0.5
    )

    EVI = safe_div(
        2.5 * (N - R),
        N + 6 * R - 7.5 * B + 1
    )

    NDWI = safe_div(G - N, G + N)

    RVI = safe_div(N, R)

    DVI = (N - R).astype(np.float32)

    OSAVI = safe_div(
        N - R,
        N + R + 0.16
    )

    index_list = [
        NDVI,
        GNDVI,
        SAVI,
        EVI,
        NDWI,
        RVI,
        DVI,
        OSAVI
    ]

    index_names = [
        "NDVI",
        "GNDVI",
        "SAVI",
        "EVI",
        "NDWI",
        "RVI",
        "DVI",
        "OSAVI"
    ]

    for arr, nm in zip(index_list, index_names):
        add(arr, nm)

    # -------------------------------------------------------------
    # LOCAL STATS
    # -------------------------------------------------------------

    for bnd, nm in [
        (N, "NIR"),
        (R, "Red"),
        (NDVI, "NDVI")
    ]:

        mu = uniform_filter(
            bnd,
            size=win
        ).astype(np.float32)

        mu2 = uniform_filter(
            bnd ** 2,
            size=win
        ).astype(np.float32)

        std = np.sqrt(
            np.clip(mu2 - mu ** 2, 0, None)
        ).astype(np.float32)

        add(mu, f"Mu_{nm}")
        add(std, f"Std_{nm}")

    # -------------------------------------------------------------
    # EDGE FEATURES
    # -------------------------------------------------------------

    for bnd, nm in [
        (N, "NIR"),
        (R, "Red")
    ]:

        b = np.nan_to_num(bnd, nan=0.0)

        sx = sp_sobel(b, axis=1)

        sy = sp_sobel(b, axis=0)

        sob = np.sqrt(sx ** 2 + sy ** 2)

        lap = np.abs(sp_laplace(b))

        add(sob, f"Sobel_{nm}")

        add(lap, f"Laplace_{nm}")

    # -------------------------------------------------------------
    # TEXTURE FEATURES
    # -------------------------------------------------------------

    for tname, tband in [
        ("NDVI", NDVI),
        ("NIR", N)
    ]:

        if tname == "NDVI":
            lo, hi = -1, 1
        else:
            lo, hi = 0, 1

        bq = (
            (
                np.clip(tband, lo, hi) - lo
            ) / (hi - lo) * 31
        ).astype(np.float32)

        for (dy, dx), dname in [
            ((0, 1), "H"),
            ((1, 0), "V")
        ]:

            J = np.roll(
                np.roll(bq, dy, axis=0),
                dx,
                axis=1
            ).astype(np.float32)

            diff = bq - J

            d2 = diff ** 2

            add(
                uniform_filter(d2, size=win),
                f"Contrast_{tname}_{dname}"
            )

            add(
                uniform_filter(np.abs(diff), size=win),
                f"Dissim_{tname}_{dname}"
            )

            add(
                uniform_filter(1.0 / (1.0 + d2), size=win),
                f"Homog_{tname}_{dname}"
            )

    # -------------------------------------------------------------
    # RATIOS
    # -------------------------------------------------------------

    add(safe_div(N, G), "Ratio_NIR_G")

    add(safe_div(N, R), "Ratio_NIR_R")

    add(
        safe_div(
            N - R,
            N + R + G + 1e-6
        ),
        "Ratio_NRG"
    )

    _FEAT_NAMES = names

    return np.stack(feats, axis=-1)

# ---------------------------------------------------------------------
# 7. LOAD SHAPEFILE
# ---------------------------------------------------------------------

print("\n📥 Loading training shapefile...")

gdf = gpd.read_file(
    CFG["shp_path"],
    engine="fiona"
)

with rasterio.open(CFG["raster_path"]) as src:

    raster_crs = src.crs

    nodata_dn = (
        src.nodata
        if src.nodata is not None
        else CFG["nodata_val"]
    )

gdf = gdf.to_crs(raster_crs)

print(f"✅ Polygons loaded: {len(gdf)}")

print(
    gdf[CFG["class_field"]]
    .value_counts()
)

# ---------------------------------------------------------------------
# 8. TRAINING SAMPLE EXTRACTION
# ---------------------------------------------------------------------

print("\n🧠 Extracting training samples...")

X_list = []
y_list = []

PAD = CFG["pad"]

t0 = time.time()

with rasterio.open(CFG["raster_path"]) as src:

    for idx, row in gdf.iterrows():

        geom = [mapping(row.geometry)]

        label = int(row[CFG["class_field"]])

        try:

            chip, chip_tf = rio_mask(
                src,
                geom,
                crop=True,
                pad=True,
                pad_width=PAD
            )

            chip = (
                chip.astype(np.float32)
                / CFG["scale_factor"]
            )

            chip = np.clip(chip, 0.0, 1.0)

            H = chip.shape[1]
            W = chip.shape[2]

            nodata_mask = np.all(
                chip != (
                    nodata_dn /
                    CFG["scale_factor"]
                ),
                axis=0
            )

            feat_chip = compute_features(
                chip,
                win=CFG["win"]
            )

            poly_mask = ~geometry_mask(
                geom,
                transform=chip_tf,
                invert=False,
                out_shape=(H, W)
            )

            valid = (
                poly_mask
                & nodata_mask
                & ~np.isnan(feat_chip).any(axis=-1)
            )

            ys, xs = np.where(valid)

            if len(ys) == 0:
                continue

            X_list.append(
                feat_chip[ys, xs, :]
            )

            y_list.append(
                np.full(
                    len(ys),
                    label,
                    dtype=np.int8
                )
            )

            if idx % 100 == 0:
                print(f"   Processed: {idx}")

        except Exception as e:

            print(f"⚠️ Polygon skipped: {e}")

# ---------------------------------------------------------------------
# 9. STACK TRAINING DATA
# ---------------------------------------------------------------------

print("\n📦 Stacking arrays...")

X = np.vstack(X_list).astype(np.float32)

y = np.concatenate(y_list)

print(f"✅ Training Pixels: {len(y):,}")

print(f"Features: {X.shape[1]}")

print(f"Extraction Time: {(time.time()-t0)/60:.1f} min")

# ---------------------------------------------------------------------
# 10. SMOTE
# ---------------------------------------------------------------------

print("\n⚖️ Applying SMOTE...")

X_bal, y_bal = SMOTE(
    random_state=CFG["seed"]
).fit_resample(X, y)

print(f"Balanced Samples: {len(y_bal):,}")

# ---------------------------------------------------------------------
# 11. STANDARDIZE
# ---------------------------------------------------------------------

print("\n📏 Scaling features...")

scaler = StandardScaler()

X_sc = scaler.fit_transform(X_bal).astype(np.float32)

# ---------------------------------------------------------------------
# 12. TRAIN RANDOM FOREST
# ---------------------------------------------------------------------

print("\n🌲 Training Random Forest...")

clf_rf = RandomForestClassifier(
    n_estimators=CFG["n_rf"],
    max_features="sqrt",
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=CFG["seed"]
)

clf_rf.fit(X_bal, y_bal)

print("✅ RF trained")

# ---------------------------------------------------------------------
# 13. TRAIN XGBOOST
# ---------------------------------------------------------------------

print("\n🚀 Training XGBoost...")

scale_pw = (
    float((y_bal == 0).sum())
    /
    max(float((y_bal == 1).sum()), 1)
)

clf_xgb = xgb.XGBClassifier(
    n_estimators=CFG["n_xgb"],
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pw,
    eval_metric="logloss",
    random_state=CFG["seed"],
    n_jobs=-1
)

clf_xgb.fit(X_sc, y_bal)

print("✅ XGBoost trained")

# ---------------------------------------------------------------------
# 14. SAVE MODEL
# ---------------------------------------------------------------------

print("\n💾 Saving model bundle...")

joblib.dump(
    {
        "rf": clf_rf,
        "xgb": clf_xgb,
        "scaler": scaler,
        "feature_names": _FEAT_NAMES,
        "cfg": CFG
    },
    str(OUT / "model_bundle.pkl")
)

print("✅ Model saved")

# ---------------------------------------------------------------------
# 15. CLASSIFICATION
# ---------------------------------------------------------------------

print("\n🛰️ Starting raster classification...")

tile_sz = CFG["tile_size"]

pad = CFG["pad"]

cls_path = str(
    OUT / "Borigumma_Paddy_Classified.tif"
)

with rasterio.open(CFG["raster_path"]) as src:

    H_full = src.height

    W_full = src.width

    base = src.profile.copy()

    nd_dn = (
        src.nodata
        if src.nodata is not None
        else CFG["nodata_val"]
    )

profile = base.copy()

profile.update({
    "count": 1,
    "dtype": "uint8",
    "nodata": 0,
    "compress": "lzw"
})

with rasterio.open(
    CFG["raster_path"]
) as src, rasterio.open(
    cls_path,
    "w",
    **profile
) as dst:

    n_ty = (H_full + tile_sz - 1) // tile_sz

    n_tx = (W_full + tile_sz - 1) // tile_sz

    total_tiles = n_ty * n_tx

    done = 0

    for ti in range(n_ty):

        for tj in range(n_tx):

            r0 = ti * tile_sz
            r1 = min(r0 + tile_sz, H_full)

            c0 = tj * tile_sz
            c1 = min(c0 + tile_sz, W_full)

            rp0 = max(r0 - pad, 0)
            rp1 = min(r1 + pad, H_full)

            cp0 = max(c0 - pad, 0)
            cp1 = min(c1 + pad, W_full)

            chip = src.read(
                window=Window(
                    cp0,
                    rp0,
                    cp1 - cp0,
                    rp1 - rp0
                )
            )

            chip = (
                chip.astype(np.float32)
                / CFG["scale_factor"]
            )

            chip = np.clip(chip, 0.0, 1.0)

            valid_pad = (
                np.all(
                    chip != (
                        nd_dn /
                        CFG["scale_factor"]
                    ),
                    axis=0
                )
            )

            feat_pad = compute_features(
                chip,
                win=CFG["win"]
            )

            dr0 = r0 - rp0
            dr1 = dr0 + (r1 - r0)

            dc0 = c0 - cp0
            dc1 = dc0 + (c1 - c0)

            feat_core = feat_pad[
                dr0:dr1,
                dc0:dc1,
                :
            ]

            valid_core = valid_pad[
                dr0:dr1,
                dc0:dc1
            ]

            cls_out = np.zeros(
                (r1-r0, c1-c0),
                dtype=np.uint8
            )

            ys, xs = np.where(valid_core)

            if len(ys) > 0:

                X_tile = feat_core[
                    ys,
                    xs,
                    :
                ]

                p_rf = clf_rf.predict_proba(
                    X_tile
                )[:, 1]

                p_xgb = clf_xgb.predict_proba(
                    scaler.transform(X_tile)
                )[:, 1]

                probs = (
                    p_rf + p_xgb
                ) / 2.0

                cls_out[ys, xs] = (
                    probs >= 0.5
                ).astype(np.uint8) + 1

            dst.write(
                cls_out,
                1,
                window=Window(
                    c0,
                    r0,
                    c1-c0,
                    r1-r0
                )
            )

            done += 1

            if done % 10 == 0 or done == total_tiles:

                print(
                    f"✅ Tiles: {done}/{total_tiles}"
                )

            del chip
            del feat_pad
            del feat_core
            del cls_out

            gc.collect()

print("\n✅ Classification completed")

print(f"\n📁 Output Raster:")
print(cls_path)

print("\n🎉 PIPELINE COMPLETE")

✅ PROJ DATABASE FOUND
PROJ PATH = d:\miniconda3\envs\crop_ai_env\Library\share\proj

🧪 Testing PROJ...
EPSG:4326
✅ All imports successful

📁 Output Folder:
E:\Rabi_Paddy_2526\LOT1\Koraput\Output

📥 Loading training shapefile...
✅ Polygons loaded: 94
class
1    56
0    38
Name: count, dtype: int64

🧠 Extracting training samples...
   Processed: 0

📦 Stacking arrays...
✅ Training Pixels: 354,879
Features: 37
Extraction Time: 1.0 min

⚖️ Applying SMOTE...
Balanced Samples: 375,688

📏 Scaling features...

🌲 Training Random Forest...
✅ RF trained

🚀 Training XGBoost...
✅ XGBoost trained

💾 Saving model bundle...
✅ Model saved

🛰️ Starting raster classification...
✅ Tiles: 10/192
✅ Tiles: 20/192
✅ Tiles: 30/192
✅ Tiles: 40/192
✅ Tiles: 50/192
✅ Tiles: 60/192
✅ Tiles: 70/192
✅ Tiles: 80/192
✅ Tiles: 90/192
✅ Tiles: 100/192
✅ Tiles: 110/192
✅ Tiles: 120/192
✅ Tiles: 130/192
✅ Tiles: 140/192
✅ Tiles: 150/192
✅ Tiles: 160/192
✅ Tiles: 170/192
✅ Tiles: 180/192
✅ Tiles: 190/192
✅ Tiles: 192/192

✅